In [ ]:
import pandas as pd
import geopandas as gpd
from census import Census
from us import states
from ipumspy import (
    IpumsApiClient,
    AggregateDataExtract,
    NhgisDataset,
)
from pathlib import Path
import glob
import zipfile
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 50) 
import gerrychain
import networkx as nx


In [38]:
p1_population_columns_2010 = {
    "P001003": "WHITE",      # White alone
    "P001004": "BLACK",      # Black or African American alone
    "P001005": "AMIN",       # American Indian and Alaska Native alone
    "P001006": "ASIAN",      # Asian alone
    "P001007": "NHPI",       # Native Hawaiian and Other Pacific Islander alone
    "P001008": "OTHER",     # Some Other Race alone
    "P001009": "2MORE",     # Two or more races
    "P001001": "TOTPOP"     #Total Population
               
}

p1_population_columns_2020 = {
    "P1_003N": "WHITE",         # White alone
    "P1_004N": "BLACK",         # Black or African American alone
    "P1_005N": "AMIN",          # American Indian and Alaska Native alone
    "P1_006N": "ASIAN",         # Asian alone
    "P1_007N": "NHPI",          # Native Hawaiian and Other Pacific Islander alone
    "P1_008N": "OTHER",     # Some Other Race alone
    "P1_009N": "2MORE",     # Two or more races
    "P1_001N" : "TOTPOP"  #Total Population        
}

In [39]:
CENSUS_KEY = "7e1b79ce2adac634987a423b6d7fb99510fee50e"
FIPS = 19

In [40]:
census_2020 = Census(
    key=CENSUS_KEY,      # We use the provided Census API key.
    year=2020    # We specify that we would like to use the 2020 Census data.
)
census_2010 = Census(
    key=CENSUS_KEY,      # We use the provided Census API key.
    year=2010    # We specify that we would like to use the 2020 Census data.
)

In [41]:
# Iowa 2020 counties
ia_counties_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/COUNTY/tl_2020_us_county.zip"
)
ia_counties_2020 = ia_counties_2020[ia_counties_2020["STATEFP"] == "19"]


# Iowa 2010 counties
ia_counties_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/COUNTY/2010/tl_2010_us_county10.zip"
)
ia_counties_2010 = ia_counties_2010[ia_counties_2010["STATEFP10"] == "19"]

In [42]:
df_tracts_2010 = census_2010.pl.get(
    ("NAME", *p1_population_columns_2010),
    geo={
        "for": "county:*",
        "in": f"state:{FIPS}",
    }, 
)

df_tracts_2010 = pd.DataFrame(df_tracts_2010).rename(
    columns={"NAME": "name", **p1_population_columns_2010}
)

In [43]:
df_tracts_2020 = census_2020.pl.get(
    ("NAME", *p1_population_columns_2020),
    geo={
        "for": "county:*",
        "in": f"state:{FIPS}",
    }, 
)

df_tracts_2020 = pd.DataFrame(df_tracts_2020).rename(
    columns={"NAME": "name", **p1_population_columns_2020}
)

In [44]:
df_tracts_2020.sort_values("county")

,name,WHITE,BLACK,AMIN,ASIAN,NHPI,OTHER,2MORE,TOTPOP,state,county
0,"Adair County, Iowa",7149.0,46.0,19.0,23.0,2.0,23.0,234.0,7496.0,19,001
1,"Adams County, Iowa",3531.0,12.0,19.0,13.0,0.0,22.0,107.0,3704.0,19,003
2,"Allamakee County, Iowa",12621.0,154.0,91.0,41.0,5.0,710.0,439.0,14061.0,19,005
3,"Appanoose County, Iowa",11690.0,75.0,24.0,70.0,7.0,40.0,411.0,12317.0,19,007
4,"Audubon County, Iowa",5469.0,17.0,5.0,5.0,1.0,30.0,147.0,5674.0,19,009
...,...,...,...,...,...,...,...,...,...,...,...
94,"Winnebago County, Iowa",9665.0,214.0,22.0,98.0,3.0,257.0,420.0,10679.0,19,189
95,"Winneshiek County, Iowa",18852.0,131.0,34.0,178.0,12.0,302.0,561.0,20070.0,19,191
96,"Woodbury County, Iowa",76794.0,5197.0,2515.0,3008.0,661.0,8155.0,9611.0,105941.0,19,193
97,"Worth County, Iowa",7041.0,60.0,8.0,32.0,1.0,40.0,261.0,7443.0,19,195


In [45]:
ia_counties_2020.sort_values("COUNTYFP")

,STATEFP,COUNTYFP,COUNTYNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
684,19,001,00465190,19001,Adair,Adair County,06,H1,G4020,None,None,None,A,1474404201,2597996,+41.3285283,-094.4781643,"POLYGON ((-94.7007 41.41758, -94.70067 41.4270..."
2602,19,003,00465191,19003,Adams,Adams County,06,H1,G4020,None,None,None,A,1096686247,5367875,+41.0216555,-094.6969059,"POLYGON ((-94.92768 41.07232, -94.92767 41.074..."
3149,19,005,00465192,19005,Allamakee,Allamakee County,06,H1,G4020,None,None,None,A,1655115638,51094064,+43.2749637,-091.3827510,"POLYGON ((-91.61091 43.42833, -91.61091 43.431..."
1247,19,007,00465193,19007,Appanoose,Appanoose County,06,H1,G4020,None,None,None,A,1287973257,49093607,+40.7442491,-092.8730649,"POLYGON ((-92.86858 40.89824, -92.8663 40.8982..."
1915,19,009,00465194,19009,Audubon,Audubon County,06,H1,G4020,None,None,None,A,1147264573,1152260,+41.6791780,-094.9043119,"POLYGON ((-95.0933 41.77694, -95.09329 41.7779..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3179,19,189,00465283,19189,Winnebago,Winnebago County,06,H1,G4020,None,None,None,A,1037261956,3182049,+43.3781243,-093.7434883,"POLYGON ((-93.97054 43.42971, -93.97053 43.430..."
1721,19,191,00465284,19191,Winneshiek,Winneshiek County,06,H1,G4020,None,None,None,A,1786666617,745729,+43.2929889,-091.8507880,"POLYGON ((-92.08001 43.42885, -92.08002 43.433..."
2339,19,193,00465285,19193,Woodbury,Woodbury County,06,H1,G4020,None,43580,None,A,2260790940,12793700,+42.3932198,-096.0532956,"POLYGON ((-96.4155 42.40034, -96.41519 42.4042..."
420,19,195,00465286,19195,Worth,Worth County,06,H1,G4020,None,32380,None,A,1036314932,4709127,+43.3734910,-093.2485332,"POLYGON ((-93.4977 43.43013, -93.49769 43.4302..."


In [46]:
merged_gdf = ia_counties_2010.merge(
    df_tracts_2010, left_on="COUNTYFP10", right_on="county", suffixes=("", "_df")
    )

merged_gdf["BLACK"] = merged_gdf["BLACK"].astype(int)
merged_gdf["WHITE"] = merged_gdf["WHITE"].astype(int)
merged_gdf["TOTPOP"] = merged_gdf["TOTPOP"].astype(int)

merged_gdf["POC"] = merged_gdf["TOTPOP"] - merged_gdf["WHITE"].astype(int)

merged_gdf.to_file("ia_files/sh_joined_2010")

In [47]:
merged_gdf = ia_counties_2020.merge(
    df_tracts_2020, left_on="COUNTYFP", right_on="county", suffixes=("", "_df")
    )

merged_gdf["BLACK"] = merged_gdf["BLACK"].astype(int)
merged_gdf["WHITE"] = merged_gdf["WHITE"].astype(int)
merged_gdf["TOTPOP"] = merged_gdf["TOTPOP"].astype(int)

merged_gdf["POC"] = merged_gdf["TOTPOP"] - merged_gdf["WHITE"].astype(int)

merged_gdf.to_file("ia_files/sh_joined_2020")

/Users/samstephenson/venvs/py314/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'name' to 'name_1'
  ogr_write(
